# Residual-stream SAEs for LFM2.5-230M

This notebook runs on Kaggle x2 T4 GPUs.

Trains two BatchTopK sparse autoencoders on the residual stream of
`LiquidAI/LFM2.5-230M`

- **GPU 1** -> hook `model.layers.6` (residual stream after layer 6, ~50% depth)
- **GPU 2** -> hook `model.layers.10` (residual stream after layer 10, ~75% depth)

Dataset: FineWeb-Edu (streamed), 1024-token contexts, ~100M tokens.

In [ ]:
SMOKE = False  # True: ~500k-token pipeline test. False: full 100M-token run.

MODEL_NAME = "LiquidAI/LFM2.5-230M"
LAYERS = [(6, 0), (10, 1)]  # (layer_index, gpu_index)
D_MODEL = 1024
EXPANSION = 16
K = 64
CONTEXT_SIZE = 1024
TRAINING_TOKENS = 500_000 if SMOKE else 100_000_000
N_CHECKPOINTS = 0 if SMOKE else 2
WORK = "/kaggle/working"

print(f"SMOKE={SMOKE}  training_tokens={TRAINING_TOKENS:,}")

In [ ]:
%pip install -q -U sae-lens "transformers>=5.4"

In [ ]:
import importlib.metadata as md

import torch

for pkg in ["sae-lens", "transformers", "transformer-lens", "torch", "datasets"]:
    try:
        print(f"{pkg}: {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg}: NOT INSTALLED")
n_gpus = torch.cuda.device_count()
print(f"GPUs: {n_gpus}", [torch.cuda.get_device_name(i) for i in range(n_gpus)])
assert n_gpus >= 2, "This notebook expects the 'GPU T4 x2' accelerator"

## Sanity checks

Before training: load the model, verify the residual stream is finite in
fp16 (T4 has no bf16 but the model was trained in bf16 so fp16 overflow is the
main risk), and verify that hooking the output
of `model.layers.N` yields exactly `hidden_states[N+1]`, i.e. the residual
stream after layer N.

In [ ]:
import gc

from transformers import AutoModelForCausalLM, AutoTokenizer

SAMPLE_TEXT = (
    "The Industrial Revolution began in Britain in the late eighteenth "
    "century and transformed manufacturing, transport, and agriculture. "
    "Steam engines, mechanized looms, and railways reshaped daily life, "
    "while cities grew rapidly around new factories. Historians debate the "
    "degree to which living standards improved for workers during the first "
    "decades of industrialization."
)


def check_dtype(dtype: torch.dtype) -> bool:
    """Return True if all hidden states are finite under this dtype."""
    model = (
        AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dtype)
        .to("cuda:0")
        .eval()
    )
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    enc = tok(SAMPLE_TEXT, return_tensors="pt").to("cuda:0")
    with torch.no_grad():
        out = model(**enc, output_hidden_states=True)
    hs = out.hidden_states
    print(f"\n--- dtype={dtype} | {len(hs)} hidden states ---")
    ok = True
    for i, h in enumerate(hs):
        finite = torch.isfinite(h).all().item()
        ok &= finite
        print(
            f"hidden_states[{i:2d}] mean_norm={h.norm(dim=-1).mean().item():9.2f} "
            f"finite={finite}"
        )

    # Hook-vs-hidden_states equivalence at layer 6 (only need it once).
    if ok:
        captured = {}

        def hook(_mod, _inp, output):
            captured["resid"] = output[0] if isinstance(output, tuple) else output

        handle = model.model.layers[6].register_forward_hook(hook)
        with torch.no_grad():
            out2 = model(**enc, output_hidden_states=True)
        handle.remove()
        assert captured["resid"].shape[-1] == D_MODEL
        assert torch.equal(captured["resid"], out2.hidden_states[7]), (
            "model.layers.6 output != hidden_states[7]"
        )
        print("hook(model.layers.6) == hidden_states[7]  [OK]")

    del model
    gc.collect()
    torch.cuda.empty_cache()
    return ok


if check_dtype(torch.float16):
    MODEL_DTYPE = "float16"
else:
    print("fp16 produced non-finite activations; falling back to fp32")
    assert check_dtype(torch.float32), "model is non-finite even in fp32?!"
    MODEL_DTYPE = "float32"
print(f"\nMODEL_DTYPE = {MODEL_DTYPE}")

## Training entrypoint

Training lives in `train_sae.py`. Run this notebook from a
working directory where that file is present. The launch cell below starts one
process per GPU.

## Launch both trainings

In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path

script_path = Path("train_sae.py")
assert script_path.exists(), "train_sae.py must be present in the notebook working directory"

procs = []
for layer, gpu in LAYERS:
    env = dict(
        os.environ,
        CUDA_VISIBLE_DEVICES=str(gpu),
        TOKENIZERS_PARALLELISM="false",
        WANDB_MODE="disabled",
    )
    log_path = Path(WORK) / f"train_layer{layer}.log"
    log_file = open(log_path, "w")
    cmd = [
        sys.executable, "train_sae.py",
        "--layer", str(layer),
        "--dtype", MODEL_DTYPE,
        "--training-tokens", str(TRAINING_TOKENS),
        "--n-checkpoints", str(N_CHECKPOINTS),
    ]
    procs.append((layer, subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT, env=env), log_file))
    print(f"launched layer {layer} on GPU {gpu} -> {log_path}")


def tail(path: Path, n: int = 3) -> str:
    try:
        lines = path.read_text(errors="replace").strip().splitlines()
        return "\n".join(lines[-n:])
    except FileNotFoundError:
        return "(no log yet)"


start = time.time()
while any(p.poll() is None for _, p, _ in procs):
    time.sleep(120)
    mins = (time.time() - start) / 60
    print(f"\n===== t+{mins:.0f} min =====")
    for layer, p, _ in procs:
        state = "running" if p.poll() is None else f"exited {p.returncode}"
        print(f"--- layer {layer} [{state}] ---")
        print(tail(Path(WORK) / f"train_layer{layer}.log"))

failed = False
for layer, p, log_file in procs:
    log_file.close()
    print(f"\nlayer {layer}: exit code {p.returncode}")
    if p.returncode != 0:
        failed = True
        print("".join(f"  {ln}\n" for ln in tail(Path(WORK) / f"train_layer{layer}.log", 60).splitlines()))
assert not failed, "at least one training process failed - see logs above"
print(f"\nboth trainings finished in {(time.time() - start) / 3600:.2f} h")

## Eval

Evaluate the trained SAEs from this notebook's output directory.


In [ ]:
import json
from pathlib import Path

import numpy as np
import torch
from datasets import load_dataset
from sae_lens import SAE
from transformers import AutoModelForCausalLM, AutoTokenizer

EVAL_LAYERS = [layer for layer, _gpu in LAYERS]
EVAL_BATCHES = 16
EVAL_BS = 8

SAE_ROOT = Path(WORK)
sae_layer6_cfg = SAE_ROOT / "sae_layer6" / "cfg.json"
if not sae_layer6_cfg.exists():
    raise FileNotFoundError(f"{sae_layer6_cfg} not found; run the training cell first")
print("SAE_ROOT =", SAE_ROOT)

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = (
    AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=getattr(torch, MODEL_DTYPE))
    .to("cuda:0")
    .eval()
)

ds = load_dataset("HuggingFaceFW/fineweb-edu", split="train", streaming=True)
ds = ds.shuffle(seed=1234, buffer_size=1000)
texts = []
it = iter(ds)
while len(texts) < EVAL_BATCHES * EVAL_BS:
    t = next(it)["text"]
    if len(t) > 2000:
        texts.append(t)

special = torch.tensor(tok.all_special_ids, device="cuda:0")
eval_summary = {}
for layer in EVAL_LAYERS:
    sae = SAE.load_from_disk(str(SAE_ROOT / f"sae_layer{layer}"), device="cuda:0")
    captured = {}

    def hook(_m, _i, output):
        captured["resid"] = output[0] if isinstance(output, tuple) else output

    handle = model.model.layers[layer].register_forward_hook(hook)

    l0s, evs, evs_with_special = [], [], []
    bos_err_ratio = []
    fired = torch.zeros(sae.cfg.d_sae, dtype=torch.bool, device="cuda:0")
    rng = np.random.default_rng(0)
    track_feats = rng.choice(sae.cfg.d_sae, size=6, replace=False)
    top_store = {int(f): [] for f in track_feats}

    with torch.no_grad():
        for b in range(EVAL_BATCHES):
            batch = texts[b * EVAL_BS : (b + 1) * EVAL_BS]
            enc = tok(
                batch,
                return_tensors="pt",
                truncation=True,
                max_length=CONTEXT_SIZE,
                padding="max_length",
            ).to("cuda:0")
            model(**enc)
            acts = captured["resid"].float()
            attn = enc["attention_mask"].bool()
            not_special = ~torch.isin(enc["input_ids"], special)

            def ev(m):
                x = acts[m]
                recon = sae.decode(sae.encode(x))
                return (1 - (x - recon).pow(2).sum() / (x - x.mean(0)).pow(2).sum()).item()

            evs.append(ev(attn & not_special))
            evs_with_special.append(ev(attn))

            x = acts[attn & not_special]
            ids = enc["input_ids"][attn & not_special]
            feats = sae.encode(x)
            l0s.append((feats > 0).float().sum(-1).mean().item())
            fired |= (feats > 0).any(0)

            x_special = acts[attn & ~not_special]
            if len(x_special) > 0:
                r_s = sae.decode(sae.encode(x_special))
                bos_err_ratio.append(
                    ((x_special - r_s).pow(2).sum() / (x - sae.decode(feats)).pow(2).sum()).item()
                )

            for f in track_feats:
                vals = feats[:, int(f)]
                top = torch.topk(vals, k=min(10, len(vals)))
                top_store[int(f)].extend(
                    (v.item(), ids[i].item())
                    for v, i in zip(top.values, top.indices)
                    if v.item() > 0
                )

    handle.remove()
    summary = {
        "hook": f"model.layers.{layer}",
        "L0": round(float(np.mean(l0s)), 2),
        "explained_variance": round(float(np.mean(evs)), 4),
        "explained_variance_incl_special": round(float(np.mean(evs_with_special)), 4),
        "special_vs_normal_sq_err_ratio": round(float(np.mean(bos_err_ratio)), 2),
        "dead_feature_fraction_eval": round(1 - fired.float().mean().item(), 4),
    }
    eval_summary[f"layer{layer}"] = summary
    print(f"\n===== layer {layer}: {summary}")
    for f, entries in top_store.items():
        entries.sort(reverse=True)
        toks = [f"{tok.decode([tid])!r}:{v:.2f}" for v, tid in entries[:8]]
        print(f"  feature {f:5d} top tokens: {', '.join(toks)}")

Path(WORK, "eval_summary_fixed.json").write_text(json.dumps(eval_summary, indent=2))
print("\nwrote", Path(WORK, "eval_summary_fixed.json"))


## Outputs

In [ ]:
for f in sorted(Path(WORK).rglob("*")):
    if f.is_file():
        print(f"{f.stat().st_size / 1e6:10.1f} MB  {f}")